In [ ]:
# loading libraries
library(readr)
library(fBasics)
library(dplyr)
library(ggplot2)
library(tidyr)
library(GGally)
library(knitr)
library(FactoMineR)
library(factoextra)
library(lmtest)

In [ ]:
# loading dataset
data_path <- "../data/dataset.csv"
df_raw <- read_csv(data_path, show_col_types = FALSE)

# data visualization
cat("N rows:", nrow(df_raw), "N Columns:", ncol(df_raw),"\n")
print("Columns name:")
names(df_raw)
head(df_raw)

# Na summary
cat("Tot NA:", sum(is.na(df_raw)), "\n")

# basic statistics
basic_stats <- basicStats(df_raw)
print(basic_stats)

In [ ]:
# Plots

# box-plot visualization - alcohol and residual sugar
# conversion from wide to long (ggplot2 works better with long)
df_long <- df_raw %>%
  dplyr::select('alcohol', 'residual sugar') %>%
  tidyr::pivot_longer(cols = everything(), names_to = "name", values_to = "value")

# rename labels
mapping <- c("alcohol" = "Alcohol Distribution (%)",
             "residual sugar" = "Residual Sugar Distribution")

df_long$name <- mapping[df_long$name]

# box-plot
ggplot(df_long, aes(y = value, x = 1)) +
  geom_boxplot(outlier.colour = "red", outlier.shape = 16, outlier.size = 2,
               fill = "lightblue") +
  facet_wrap(~name, scales = "free_y") +  
  labs(x = "", y = "") +
  theme_minimal() +
  theme(
    axis.text.x = element_blank(),
    axis.ticks.x = element_blank(),
    strip.text = element_text(face = "bold", size = 12)
  ) 

# qq-plot visualization - alcohol and residual sugar
# filter on residual sugar
df_rs <- df_raw[["residual sugar"]]

# qq-plot
ggplot(data.frame(residual_sugar = df_rs), aes(sample = residual_sugar)) +
  stat_qq(color = "blue") +       
  stat_qq_line(color = "red") + 
  theme_minimal() +
  labs(title = "Q-Q Plot for Residual Sugar",
       x = "",
       y = "") +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5)
  )

# histogram plot visualization - volatile acidity
# filter on volatile acidity
df_va <- df_raw[["volatile acidity"]]

# histogram plot
ggplot(data.frame(volatile_acidity = df_va), aes(x = volatile_acidity)) +
  geom_histogram(bins = 30,   
                 fill = "skyblue", 
                 color = "black") +
  labs(title = "Volatile Acidity",
       x = "volatile acidity",
       y = "Freq") +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5)
  )

# histogram plot with density curve
ggplot(data.frame(volatile_acidity = df_va), aes(x = volatile_acidity)) +
  geom_histogram(aes(y = ..density..), 
                 bins = 30,          
                 fill = "grey", 
                 color = "black",
                 alpha = 0.6) +         
  geom_density(color = "red", linewidth = 1) +
  labs(title = "Histogram with Density Curve",
       x = "volatile acidity",
       y = "Density") +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5)
  )

In [ ]:
# Exploratory Data Analysis
# delete quality column
df_pp <- df_raw %>% dplyr::select(-'quality')

# func definition with 2 arg and others (...)
my_corr <- function(data, mapping, ...) {

  # extracts numeric values
  x <- GGally::eval_data_col(data, mapping$x)
  y <- GGally::eval_data_col(data, mapping$y)
  
  # calculates corr
  test <- cor.test(x, y)

  # extracts r and p-value
  r <- round(test$estimate, 2)
  p <- test$p.value

  # defines * for p-values
  sig <- ifelse(p < 0.001, "***",
                ifelse(p < 0.01, "**",
                       ifelse(p < 0.05, "*", "")))
    
    # insert text * in the plot
    ggally_text(
    label = paste0("", r, "\n",sig),
    mapping = mapping,
    color="red",
    ...
  ) + theme_void()
}

# histogram personalization
my_diag <- function(data, mapping, ...) {
  ggplot(data = data, mapping = mapping) +
    geom_histogram(aes(y = ..density..), fill = "grey", color = "black", bins=5, alpha = 0.6) +
    geom_density(color = "red", size = 1) +    # red curve
    theme_minimal()
}

# scatterplot personalization
my_lower <- function(data, mapping, ...) {
  ggplot(data = data, mapping = mapping) +
    geom_point(alpha = 0.6) +
    geom_smooth(method = "lm", color = "red", se = FALSE) +  
    theme_minimal()
}

# pairplot
ggpairs(df_pp,
        lower = list(continuous = my_lower),      # scatterplot
        diag  = list(continuous = my_diag),       # histogram with density
        upper = list(continuous = my_corr))       # corr

# heatmap visualization
# calculates corr_matrix for heatmap use
corr_mat <- cor(df_pp, use = "pairwise.complete.obs")

# from wide to long
cor_long <- corr_mat %>%
  as.data.frame() %>%
  tibble::rownames_to_column(var = "Var1") %>%
  pivot_longer(-Var1, names_to = "Var2", values_to = "Correlation")

# 4. Heatmap
ggplot(cor_long, aes(x = Var1, y = Var2, fill = Correlation)) +
  geom_tile() +
  scale_fill_gradient2(low = "blue", mid = "white", high = "red", midpoint = 0) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Correlation Heatmap", x="",y="",fill = "r") +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5)
  )
kable(round(corr_mat, 2), caption = "Correlation Matrix")

In [ ]:
# Principal Component Analysis
# calculates pca, with centred and standardized values (scale.unit=T),
# returns ncp components, with no graphs (graph=F)
var.pca <- FactoMineR::PCA(df_pp, scale.unit = TRUE, ncp = 10, graph = FALSE)

# prints components with eigen, variance and cumulative
eig_df <- as.data.frame(var.pca$eig)
print(eig_df)    

# screeplot
factoextra::fviz_eig(var.pca, addlabels = TRUE) + 
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5)
  )

# var contribution
# var and contrib are two columns of df's PCA
var_con <- as.data.frame(var.pca$var$contrib) %>%
  tibble::rownames_to_column(var = "Variable") %>%  
  dplyr::select(Variable, Dim.1, Dim.2, Dim.3)
print(var_con)

# var contribution plot (wide to long)
var_con_long <- var_con %>%
  pivot_longer(cols = starts_with("Dim"), 
               names_to = "Dimension", 
               values_to = "Contribution")

# contribution plot
ggplot(var_con_long, aes(x = Variable, y = Contribution, fill = Dimension)) +
  geom_bar(stat = "identity", position = "dodge") +  
  labs(title = "Variable Contribution to the First 3 Components",
       x = "Variables",
       y = "Contribution (%)") +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5)
  ) +
  scale_fill_brewer(palette = "Set2")

# var coordinates
# var and coord are two columns of df's PCA
var_coord <- as.data.frame(var.pca$var$coord) %>%
  dplyr::select(Dim.1, Dim.2, Dim.3) %>%
  tibble::rownames_to_column(var = "Variable")
print(var_coord)

# correlation circle
fviz_pca_var(var.pca, col.var = "red",
             repel = TRUE,        # try to no overlay labels
             arrowsize = 1.2,
             title = "Correlation Circle - PCA") + 
             theme(
               plot.title = element_text(face = "bold", size = 14, hjust = 0.5))

# quality representation
# var and cos2 are two columns of df's PCA
var_cos2 <- as.data.frame(var.pca$var$cos2) %>%
  dplyr::select(Dim.1, Dim.2, Dim.3) %>%
  tibble::rownames_to_column(var = "Variable")
print(var_cos2)

var_cos2_long <- var_cos2 %>%
  pivot_longer(cols = starts_with("Dim"),
               names_to = "Dimension",
               values_to = "Cos2")

# quality representation plot
ggplot(var_cos2_long, aes(x = Variable, y = Cos2, fill = Dimension)) +
  geom_bar(stat = "identity", position = "dodge") +  
  labs(title = "Variable representation quality (cos²)",
       x = "Variable",
       y = expression(cos^2)) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5)
  ) +
  scale_fill_brewer(palette = "Set2")

In [ ]:
# Linear regression
# linear model construction
# quality ~ . says to R to use other columns as independent and
# quality as dependent. cbind() combines in a new df: lm() expects
# that x and y are in the same df
mod <- lm(quality ~ ., data = cbind(quality = df_raw$quality, df_pp))

# printing
summary(mod)

# backward selection algorithm to reduce var
mod_r <- step(mod, direction = "both")
summary(mod_r)

In [ ]:
# Model validation

# t test
res = residuals(mod_r)
# t test res mean = 0
t_test <- t.test(res, mu = 0)
print(t_test)

# shapiro-wilk test
shapiro = shapiro.test(res)
print(shapiro) 

# shapiro qqplot
qqnorm(res, main = "Residuals Q-Q Plot", col="blue") 
qqline(res, col = "red", lwd = 2)

# Breusch-Pagan test
bp = bptest(mod_r) 
print(bp)

# Durbin-Watson test
dw = dwtest(mod_r)
print(dw)

In [ ]:
# residuals analysis

# res plot
fit = fitted(mod_r)
plot(fit, res,
     xlab = "Fitted values",
     ylab = "Residuals",
     main = "Residuals vs Fitted",
     pch = 19, col = "darkblue")

abline(h = 0, col = "green", lwd = 2)
abline(h =mean(res), col = "red", lty = 2)

# histogram res plot
ggplot(data.frame(res), aes(x = res)) +
  geom_histogram(bins = 30,   
                 fill = "skyblue", 
                 color = "black") +
  labs(title = "Residual Distribution Histogram",
       x = "Residuals",
       y = "Freq") +
  theme_minimal() +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5)
  )

# res density plot
ggplot(data.frame(res), aes(x = res)) +
  geom_density(color="blue", size=1) +
  stat_function(
      fun = dnorm,
      args = list(mean = mean(res,na.rm=T),
                  sd = sd(res, na.rm=T)),
      color = "red", linetype = "dashed", size = 1
      ) +
      labs(title = "Residual Distribution",
           x = "Residuals",
           y = "Density") +
      theme_minimal() +
      theme(plot.title = element_text(face = "bold", size = 14, hjust = 0.5))

# residuals homoscedasticity
res_sd = abs(res)

# res plot
plot(x = fit, 
     y = res_sd,
     main = "Standardized Residuals vs Fitted Values",
     xlab = "Fitted Values",
     ylab = "Standardized Residuals",
     pch = 20, 
     col = "blue")
abline(h = 0, col = "red", lty = 2)

In [ ]:
# Outliers

# leverage score
leverage <- hatvalues(mod_r)

# leverage plot
plot(x = 1:length(leverage),
     y = leverage,
     main = "Leverage Scores vs Observation Index",
     xlab = "Index",
     ylab = "Leverage Score",
     pch = 19)

# limit is 2 * (p+1) / n,
p <- length(mod_r$coefficients) - 1
n <- length(leverage)
ll <- 2 * (p+1) / n
abline(h = ll, col = "red", lty = 2)

# Cook's D Chart
# outliers / influential observations
cooks <- cooks.distance(mod_r)

plot(x = 1:length(cooks),
     y = cooks,
     main = "Cook's D Chart",
     xlab = "Observation",
     ylab = "Cook's Distance",
     pch = 19,
     col = "light blue")

# limit is 4 / n
lc = 4 / length(cooks)
print(paste("threshold is: ",lc))
abline(h = lc, col = "red", lty = 2, lwd = 2)

cookst <- data.frame(
  Observation = 1:length(cooks),
  Cooks_Distance = cooks) %>% 
       filter(Cooks_Distance > lc) %>%
       arrange(desc(Cooks_Distance))
head(cookst,20)

In [ ]:
# New valuation model

# outliers index
idx <- cookst$Observation

# new df
X_clean <- df_pp[-idx, ]
y_clean <- df_raw$quality[-idx]

# new model valuation
mod_clean <- lm(y_clean ~ ., data = X_clean)
summary(mod_clean)